# c_prototipo_desplegable

Prototipo interactivo en formato notebook para consumir la API del Sprint 6.

Este notebook no reemplaza la API FastAPI ni el dashboard Streamlit. Funciona como una interfaz academica/desplegable en entornos Jupyter/AWS compatibles con `ipywidgets`.

## Uso esperado

1. Levantar la API PB-19 localmente o usar el endpoint desplegado en AWS.
2. Configurar `API_URL` en el widget del notebook.
3. Probar `/health` y `/version`.
4. Enviar una sesion de visitante a `/predict`.

Endpoint local por defecto: `http://localhost:8000`.

In [ ]:
import json
import os
from pathlib import Path

import requests
import ipywidgets as widgets
from IPython.display import HTML, display

PROJECT_NAME = "Proyecto 2 - Conversion de Visitantes Online"
DEFAULT_API_URL = os.getenv("API_URL", "http://localhost:8000")
EXAMPLE_REQUEST_PATH = Path("handoff/contracts/c_example_request.json")

if EXAMPLE_REQUEST_PATH.exists():
    DEFAULT_PAYLOAD = json.loads(EXAMPLE_REQUEST_PATH.read_text(encoding="utf-8"))
else:
    DEFAULT_PAYLOAD = {
        "Administrative": 0,
        "Administrative_Duration": 0.0,
        "Informational": 0,
        "Informational_Duration": 0.0,
        "ProductRelated": 1,
        "ProductRelated_Duration": 0.0,
        "BounceRates": 0.2,
        "ExitRates": 0.2,
        "PageValues": 0.0,
        "SpecialDay": 0.0,
        "Month": "Feb",
        "OperatingSystems": 1,
        "Browser": 1,
        "Region": 1,
        "TrafficType": 1,
        "VisitorType": "Returning_Visitor",
        "Weekend": False,
    }

display(HTML(f"<h2>{PROJECT_NAME}</h2><p>Prototipo desplegable via notebook + API Sprint 6.</p>"))

In [ ]:
api_url = widgets.Text(
    value=DEFAULT_API_URL,
    description="API_URL",
    layout=widgets.Layout(width="70%"),
)

health_button = widgets.Button(description="Probar /health", button_style="info")
version_button = widgets.Button(description="Probar /version", button_style="info")
connection_output = widgets.Output()


def normalize_base_url() -> str:
    return api_url.value.rstrip("/")


def call_get(endpoint: str):
    response = requests.get(f"{normalize_base_url()}{endpoint}", timeout=10)
    response.raise_for_status()
    return response.json()


def on_health_click(_):
    with connection_output:
        connection_output.clear_output()
        try:
            print(json.dumps(call_get("/health"), indent=2, ensure_ascii=False))
        except Exception as exc:
            print(f"Error consultando /health: {exc}")


def on_version_click(_):
    with connection_output:
        connection_output.clear_output()
        try:
            print(json.dumps(call_get("/version"), indent=2, ensure_ascii=False))
        except Exception as exc:
            print(f"Error consultando /version: {exc}")


health_button.on_click(on_health_click)
version_button.on_click(on_version_click)

display(widgets.VBox([
    widgets.HTML("<h3>Conexion con la API</h3>"),
    api_url,
    widgets.HBox([health_button, version_button]),
    connection_output,
]))

In [ ]:
def make_float_widget(key, description, min_value=0.0, max_value=1000.0, step=1.0):
    return widgets.FloatText(value=float(DEFAULT_PAYLOAD[key]), description=description, layout=widgets.Layout(width="45%"))


fields = {
    "Administrative": widgets.IntText(value=int(DEFAULT_PAYLOAD["Administrative"]), description="Pag. admin"),
    "Administrative_Duration": make_float_widget("Administrative_Duration", "T. admin"),
    "Informational": widgets.IntText(value=int(DEFAULT_PAYLOAD["Informational"]), description="Pag. info"),
    "Informational_Duration": make_float_widget("Informational_Duration", "T. info"),
    "ProductRelated": widgets.IntText(value=int(DEFAULT_PAYLOAD["ProductRelated"]), description="Pag. prod"),
    "ProductRelated_Duration": make_float_widget("ProductRelated_Duration", "T. prod"),
    "BounceRates": widgets.FloatSlider(value=float(DEFAULT_PAYLOAD["BounceRates"]), min=0, max=1, step=0.01, description="Rebote"),
    "ExitRates": widgets.FloatSlider(value=float(DEFAULT_PAYLOAD["ExitRates"]), min=0, max=1, step=0.01, description="Salida"),
    "PageValues": make_float_widget("PageValues", "Valor pag."),
    "SpecialDay": widgets.FloatSlider(value=float(DEFAULT_PAYLOAD["SpecialDay"]), min=0, max=1, step=0.01, description="Fecha esp."),
    "Month": widgets.Dropdown(options=["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], value=DEFAULT_PAYLOAD["Month"], description="Mes"),
    "OperatingSystems": widgets.IntText(value=int(DEFAULT_PAYLOAD["OperatingSystems"]), description="SO"),
    "Browser": widgets.IntText(value=int(DEFAULT_PAYLOAD["Browser"]), description="Browser"),
    "Region": widgets.IntText(value=int(DEFAULT_PAYLOAD["Region"]), description="Region"),
    "TrafficType": widgets.IntText(value=int(DEFAULT_PAYLOAD["TrafficType"]), description="Trafico"),
    "VisitorType": widgets.Dropdown(options=["Returning_Visitor", "New_Visitor", "Other"], value=DEFAULT_PAYLOAD["VisitorType"], description="Visitante"),
    "Weekend": widgets.Checkbox(value=bool(DEFAULT_PAYLOAD["Weekend"]), description="Fin de semana"),
}

predict_button = widgets.Button(description="Predecir compra", button_style="success")
prediction_output = widgets.Output()


def build_payload() -> dict:
    return {key: widget.value for key, widget in fields.items()}


def classify_probability(probability: float) -> str:
    if probability >= 0.70:
        return "Alta intencion: activar cupon, chat u oferta personalizada."
    if probability >= 0.35:
        return "Intencion media: mostrar recomendacion o recordatorio."
    return "Baja intencion: mantener navegacion normal o retargeting posterior."


def on_predict_click(_):
    with prediction_output:
        prediction_output.clear_output()
        payload = build_payload()
        try:
            response = requests.post(f"{normalize_base_url()}/predict", json=payload, timeout=15)
            response.raise_for_status()
            result = response.json()
            probability = float(result["purchase_probability"])
            prediction = int(result["prediction"])
            display(HTML(
                f"<h3>Resultado</h3>"
                f"<p><b>Probabilidad de compra:</b> {probability:.2%}</p>"
                f"<p><b>Prediccion:</b> {'Compra probable' if prediction == 1 else 'Compra no priorizada'}</p>"
                f"<p><b>Threshold:</b> {result.get('threshold')}</p>"
                f"<p>{classify_probability(probability)}</p>"
            ))
            print(json.dumps(result, indent=2, ensure_ascii=False))
        except Exception as exc:
            print(f"Error consultando /predict: {exc}")


predict_button.on_click(on_predict_click)

left = widgets.VBox([
    fields["Administrative"], fields["Administrative_Duration"],
    fields["Informational"], fields["Informational_Duration"],
    fields["ProductRelated"], fields["ProductRelated_Duration"],
])
right = widgets.VBox([
    fields["BounceRates"], fields["ExitRates"], fields["PageValues"], fields["SpecialDay"],
    fields["Month"], fields["OperatingSystems"], fields["Browser"], fields["Region"],
    fields["TrafficType"], fields["VisitorType"], fields["Weekend"],
])

display(widgets.VBox([
    widgets.HTML("<h3>Simulador interactivo de visitante</h3>"),
    widgets.HBox([left, right]),
    predict_button,
    prediction_output,
]))

## Notas para AWS/Jupyter

- Si la API esta en EC2, reemplazar `API_URL` por `http://IP_PUBLICA:8000` o por el dominio configurado.
- Si se usa SageMaker Studio o JupyterLab en AWS, asegurar que `ipywidgets` este habilitado.
- Este notebook consume la API PB-19; no reentrena modelos y no modifica artefactos.
- Para PB-21, el dashboard final puede usar el mismo contrato documentado en `handoff/contracts/api_contract.md`.